In this notebook, we will:
1. Download all available posters from our database with 5000 movies
2. Extract poster embeddings using CLIP
3. Try different approaches to find meaningful poster clusters
4. Examine potential relations between poster clusters and other variables (countries, decades, etc.)


In [7]:
from pathlib import Path
import pandas as pd
from tqdm import tqdm
import json
import requests

# PHASE 4

# Posters download

#  - read the Phase 3 output CSV
#  - construct full TMDb URLs from relative poster paths
#  - save with the correct file extension (jpg)
#  - skip files already saved and keep a checkpoint of downloaded IDs
# 

# configuration

CSV_PATH        = Path("horror_data/horror_categorized_FULLSAMPLE_clean.csv")
OUT_DIR         = Path("artifacts_posters")
POSTERS_DIR     = OUT_DIR / "posters"
CHECKPOINT_JSON = OUT_DIR / "_download_checkpoint.json"
TMDB_BASE       = "https://image.tmdb.org/t/p/original"
COL_ID          = "id"
COL_POSTER      = "poster_path"

# creating the output directories as folders (after checking if they exist already)
OUT_DIR.mkdir(parents=True, exist_ok=True)
POSTERS_DIR.mkdir(parents=True, exist_ok=True)

# ---- load data ----
df = pd.read_csv(CSV_PATH)
df = df[df[COL_POSTER].notna()].copy()

# construct full URL and local path
df["poster_url"]  = TMDB_BASE.rstrip("/") + df[COL_POSTER]
df["poster_file"] = df[COL_ID].astype(str) + ".jpg"
df["local_path"]  = df["poster_file"].apply(lambda s: POSTERS_DIR / s)

# ---- load checkpoint ----
if CHECKPOINT_JSON.exists():
    try:
        checkpoint = set(json.loads(CHECKPOINT_JSON.read_text()).get("downloaded_ids", []))
    except Exception:
        checkpoint = set()
else:
    checkpoint = set()

# ---- download loop ----
downloaded_ids = set(checkpoint)
for _, row in tqdm(df.iterrows(), total=len(df), desc="Downloading posters"):
    mid = str(row[COL_ID])
    if mid in downloaded_ids:
        continue  # skip already done

    url = row["poster_url"]
    out_path = row["local_path"]

    # skip if file exists (from previous run)
    if out_path.exists():
        downloaded_ids.add(mid)
        continue

    try:
        r = requests.get(url, timeout=(5, 30))
        if r.status_code == 200:
            with open(out_path, "wb") as f:
                f.write(r.content)
            downloaded_ids.add(mid)
        else:
            print(f"HTTP {r.status_code} for {mid}")
    except Exception as e:
        print(f"Error downloading {mid}: {e}")

    # update checkpoint every 50 images
    if len(downloaded_ids) % 50 == 0:
        CHECKPOINT_JSON.write_text(json.dumps({"downloaded_ids": sorted(list(downloaded_ids))}, indent=2))

# ---- final checkpoint save ----
CHECKPOINT_JSON.write_text(json.dumps({"downloaded_ids": sorted(list(downloaded_ids))}, indent=2))

print(f"Done. Posters OK: {len(downloaded_ids)} / {len(df)}")
print(f"Saved under: {POSTERS_DIR.resolve()}")

Done. Posters OK: 4860 / 4860
Saved under: /Users/lara/dataviz-s1/machine-learning/Project/artifacts_posters/posters


In [2]:
# CLIP embeddings extraction for posters
# --------------------------------------------
# - reads all image files in POSTERS_DIR
# - uses the CLIP model to transform each poster image into a high-dimensional embedding vector that captures semantic visual content
# - outputs:
#     artifacts_posters/clip_embeddings.npy  
#     artifacts_posters/embeddings_index.json (metadata list: movie_id, file_name)

import torch
import numpy as np
from PIL import Image
from pathlib import Path
from transformers import CLIPProcessor, CLIPModel
import json
from tqdm import tqdm

# ----------------------
# config
# ----------------------
# model_name selects the exact CLIP variant (on this case vit-base-patch32)
# posters_dir is the folder where the image files are
# out_dir is where all resulting artifacts will be saved
model_name = "openai/clip-vit-base-patch32"
POSTERS_DIR = Path("artifacts_posters/posters")
OUT_DIR = Path("artifacts_posters")
EMBEDDINGS_FILE = OUT_DIR / "clip_embeddings.npy"
EMBEDDINGS_INDEX = OUT_DIR / "embeddings_index.json"

# creating output directory if missing ensures the script works on clean environments
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ----------------------
# load model & processor
# ----------------------
print("Loading CLIP model...")

device = "cuda" if torch.cuda.is_available() else "cpu"
# CLIPModel loads the pretrained weights
model = CLIPModel.from_pretrained(model_name).to(device)

# CLIPProcessor handles all preprocessing (resize, normalization, etc.) consistently
processor = CLIPProcessor.from_pretrained(model_name)

# ----------------------
# helper: single-image embedding
# ----------------------
def extract_clip_embedding(image_path: Path) -> np.ndarray | None:
    """
    extract CLIP image embedding for a single poster.

    returns:
        a normalized embedding vector (1d numpy array) or None if something fails
    """
    try:
        image = Image.open(image_path).convert("RGBA")

        # processor applies CLIP-specific transforms and returns tensors ready for the model
        inputs = processor(images=image, return_tensors="pt").to(device)

        # CLIP's image encoder converts image to semantic vector
        image_features = model.get_image_features(**inputs)
        image_features = torch.nn.functional.normalize(image_features, p=2, dim=-1)

        embedding = image_features.detach().cpu().numpy().reshape(-1)

        return embedding

    except Exception as e:
        # errors are caught instead of breaking the pipeline
        print(f"Error processing {image_path}: {e}")
        return None

# ----------------------
# main: loop over posters
# ----------------------
print("Collecting poster files...")

# gathering all possible image types
poster_files = []
for ext in ("*.jpg", "*.jpeg", "*.png", "*.webp"):
    poster_files.extend(POSTERS_DIR.glob(ext))

poster_files = sorted(poster_files)

print(f"Found {len(poster_files)} poster files.")

embeddings_list = []
embeddings_index = []

for poster_path in tqdm(poster_files, desc="Extracting CLIP embeddings"):
    emb = extract_clip_embedding(poster_path)
    if emb is None:
        # skipping missing/failed embeddings avoids breaking the alignment between features and metadata
        continue

    # append the embedding
    embeddings_list.append(emb)

    # storing metadata (movie_id and file_name) lets you map rows to movies after saving the matrix
    embeddings_index.append({
        "movie_id": poster_path.stem,
        "file_name": poster_path.name,
    })

# ----------------------
# save outputs
# ----------------------
# sanity check: if no embeddings, something is wrong
if len(embeddings_list) == 0:
    raise RuntimeError("No embeddings were generated. Check POSTERS_DIR and file types.")

# vstack converts a list of 1d arrays into a single 2d matrix
embeddings_matrix = np.vstack(embeddings_list)

np.save(EMBEDDINGS_FILE, embeddings_matrix)

# json preserving row to file metadata alignment
with open(EMBEDDINGS_INDEX, "w") as f:
    json.dump(embeddings_index, f, indent=2)

print(f"Saved embeddings matrix to: {EMBEDDINGS_FILE}")
print(f"Matrix shape: {embeddings_matrix.shape}")
print(f"Saved index JSON to:       {EMBEDDINGS_INDEX}")

Loading CLIP model...
Found 4860 poster files.


Extracting CLIP embeddings: 100%|██████████| 4860/4860 [04:15<00:00, 19.03it/s]

Saved embeddings matrix to: artifacts_posters/clip_embeddings.npy
Matrix shape: (4860, 512)
Saved index JSON to:       artifacts_posters/embeddings_index.json


In [12]:
# ============================================================
# EXPERIMENTS WITH KMEANS CLUSTERING ON PCA-REDUCED EMBEDDINGS
# ============================================================

import json
import random
from math import ceil
from collections import defaultdict
from pathlib import Path

import numpy as np
from PIL import Image
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

# ------------------------------------------------------------
# global paths shared by all experiments
# ------------------------------------------------------------

# base_dir is the root folder where everything related to posters lives
BASE_DIR = Path("artifacts_posters")

# posters_dir holds the original poster image files (jpg/png/webp)
POSTERS_DIR = BASE_DIR / "posters"

# clip_emb_path is the single .npy file with 512-dimensional CLIP embeddings
CLIP_EMB_PATH = BASE_DIR / "clip_embeddings.npy"

# global_index_json is the master JSON where each row of CLIP_EMB_PATH
# is mapped to some metadata such as {movie_id, file_name}
GLOBAL_INDEX_JSON = BASE_DIR / "embeddings_index.json"


# ------------------------------------------------------------
# helper: build collage images for each cluster
# ------------------------------------------------------------
def build_cluster_galleries(
    index_with_clusters,
    posters_dir: Path,
    cluster_gallery_dir: Path,
    max_per_cluster: int = 36,
    thumb_w: int = 256,
    thumb_h: int = 384,
    grid_cols: int = 6,
):
    """
    given a list of metadata records that already have a "cluster" field,
    this function creates one collage image per cluster.

    each collage is composed of thumbnails of posters that belong to that cluster.
    the collages are saved as "cluster_<id>.jpg" inside cluster_gallery_dir.
    """

    # make sure the gallery folder exists so saves don't fail
    cluster_gallery_dir.mkdir(parents=True, exist_ok=True)

    # group all records by cluster id in a dict: cluster_id -> list[record]
    clusters = defaultdict(list)
    for info in index_with_clusters:
        # we use get(..., -1) so missing "cluster" defaults to -1
        cid = info.get("cluster", -1)
        # we treat cluster -1 as "noise" and skip it
        if cid == -1:
            continue
        clusters[cid].append(info)

    # now we iterate over each cluster and create one collage per cluster
    for cid, items in clusters.items():
        # in case a cluster has many posters, we cap the number of thumbnails
        # to keep the collage readable and the file size reasonable
        if len(items) <= max_per_cluster:
            sample = items
        else:
            sample = random.sample(items, max_per_cluster)

        n = len(sample)

        # compute how many rows we need in the grid based on how many
        # thumbnails we will place and how many columns per row
        rows = ceil(n / grid_cols)

        # compute total collage width and height in pixels
        gallery_w = grid_cols * thumb_w
        gallery_h = rows * thumb_h

        # create a blank black canvas where all thumbnails will be pasted
        gallery = Image.new("RGB", (gallery_w, gallery_h), color=(0, 0, 0))

        # loop over each poster sample and paste it in the correct grid cell
        for i, info in enumerate(sample):
            poster_path = posters_dir / info["file_name"]
            if not poster_path.exists():
                # if the file is missing we skip it
                continue

            # open the image and ensure it is in RGB mode
            img = Image.open(poster_path).convert("RGB")

            # resize the image in-place so that it fits inside the thumbnail box
            # while preserving aspect ratio
            img.thumbnail((thumb_w, thumb_h), Image.LANCZOS)

            # create a fixed-size thumbnail canvas so every slot has the same size
            thumb = Image.new("RGB", (thumb_w, thumb_h), color=(0, 0, 0))

            # center the resized image within that thumbnail box
            offset_x = (thumb_w - img.width) // 2
            offset_y = (thumb_h - img.height) // 2
            thumb.paste(img, (offset_x, offset_y))

            # compute the grid coordinates: column and row index
            col = i % grid_cols
            row = i // grid_cols

            # convert grid coordinates into pixel coordinates
            x = col * thumb_w
            y = row * thumb_h

            # paste the thumbnail into the collage canvas
            gallery.paste(thumb, (x, y))

        # build the output path for this cluster's collage
        out_path = cluster_gallery_dir / f"cluster_{cid}.jpg"

        # save the collage as a jpg file
        gallery.save(out_path, quality=90)
        print(f"    saved gallery for cluster {cid} -> {out_path}")


# ------------------------------------------------------------
# single kmeans + PCA experiment
# ------------------------------------------------------------
def run_kmeans_experiment(PCA_DIM: int, K: int):
    """
    runs one complete experiment with the following steps:

    1) load the global 512-dimensional CLIP embeddings and their index
    2) reduce the embeddings to PCA_DIM dimensions using PCA
    3) run kmeans with K clusters on the PCA-reduced embeddings
    4) attach the cluster label to each metadata record as "cluster"
    5) save:
         - the pca embeddings for this experiment
         - the enriched metadata with cluster labels
         - a collage image for every cluster
    """

    experiment_name = f"kmeans_k{K}_pca{PCA_DIM}"

    # all the artifacts for this experiment live under this folder
    exp_dir = BASE_DIR / "experiments" / experiment_name
    exp_dir.mkdir(parents=True, exist_ok=True)

    # path where we will save the experiment-specific index with cluster labels
    index_with_clusters_path = exp_dir / "embeddings_with_clusters.json"

    # path where we will save the pca-reduced embeddings
    pca_embeddings_path = exp_dir / f"clip_embeddings_pca{PCA_DIM}d.npy"

    # folder where we will store collage images, one per cluster
    cluster_gallery_dir = exp_dir / "cluster_galleries"
    cluster_gallery_dir.mkdir(parents=True, exist_ok=True)

    print("\n====================================================")
    print("kmeans experiment:", experiment_name)
    print("  pca_dim:", PCA_DIM, "k:", K)
    print("  output folder:", exp_dir)

    # -----------------------
    # step 1: load embeddings and index
    # -----------------------

    # load the global 512d CLIP embeddings into a numpy array
    emb = np.load(CLIP_EMB_PATH)

    # load the metadata index so we can associate each embedding with a file_name
    with GLOBAL_INDEX_JSON.open() as f:
        index = json.load(f)

    # -----------------------
    # step 2: apply pca
    # -----------------------

    # create a pca object that will project the original 512d vectors into
    # a lower-dimensional space of size PCA_DIM
    reducer = PCA(n_components=PCA_DIM)

    # fit the pca model on the embeddings and transform them in one go
    emb_pca = reducer.fit_transform(emb)

    # save the reduced embeddings to reuse later for analysis
    np.save(pca_embeddings_path, emb_pca)
    print("  saved pca embeddings to:", pca_embeddings_path)

    # -----------------------
    # step 3: kmeans clustering
    # -----------------------

    # create a kmeans model with K clusters; random_state is fixed so that
    # rerunning the same experiment gives the same assignments
    kmeans = KMeans(n_clusters=K, random_state=14)

    # fit kmeans on the pca embeddings and get the cluster label for each point
    labels = kmeans.fit_predict(emb_pca)

    # -----------------------
    # step 4: attach labels to metadata
    # -----------------------

    # each record in "index" corresponds to one embedding and now receives
    # an integer "cluster" field with its kmeans cluster id
    for rec, lab in zip(index, labels):
        rec["cluster"] = int(lab)

    # -----------------------
    # step 5: save index + galleries
    # -----------------------

    # write the enriched index with cluster labels
    with index_with_clusters_path.open("w") as f:
        json.dump(index, f, indent=2, ensure_ascii=False)
    print("  wrote enriched metadata with clusters to:", index_with_clusters_path)

    # build collage images for each cluster using the enriched index
    build_cluster_galleries(index, POSTERS_DIR, cluster_gallery_dir)



In [13]:
# ------------------------------------------------------------
# list of kmeans + pca variants to run
# ------------------------------------------------------------
EXPERIMENTS = [
    # experiment 1: 240 pca, 20 clusters
    {"PCA_DIM": 240, "K": 20},

    # experiment 2: 280 pca, 50 clusters
    {"PCA_DIM": 280, "K": 50},

    # experiment 3: 120 pca, 20 clusters
    {"PCA_DIM": 120, "K": 20},

    # experiment 4: 160 pca, 40 clusters
    {"PCA_DIM": 160, "K": 40},

]

# ------------------------------------------------------------
# run all defined experiments
# ------------------------------------------------------------

for cfg in EXPERIMENTS:
    # unpack the dictionary keys to arguments and call the experiment function
    run_kmeans_experiment(**cfg)


kmeans experiment: kmeans_k20_pca240
  pca_dim: 240 k: 20
  output folder: artifacts_posters/experiments/kmeans_k20_pca240
  saved pca embeddings to: artifacts_posters/experiments/kmeans_k20_pca240/clip_embeddings_pca240d.npy
  wrote enriched metadata with clusters to: artifacts_posters/experiments/kmeans_k20_pca240/embeddings_with_clusters.json


/Users/lara/.pyenv/versions/3.11.8/lib/python3.11/site-packages/sklearn/utils/extmath.py:350: RuntimeWarning: divide by zero encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/lara/.pyenv/versions/3.11.8/lib/python3.11/site-packages/sklearn/utils/extmath.py:350: RuntimeWarning: overflow encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/lara/.pyenv/versions/3.11.8/lib/python3.11/site-packages/sklearn/utils/extmath.py:350: RuntimeWarning: invalid value encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/lara/.pyenv/versions/3.11.8/lib/python3.11/site-packages/sklearn/utils/extmath.py:351: RuntimeWarning: divide by zero encountered in matmul
  Q, _ = normalizer(A.T @ Q)
/Users/lara/.pyenv/versions/3.11.8/lib/python3.11/site-packages/sklearn/utils/extmath.py:351: RuntimeWarning: overflow encountered in matmul
  Q, _ = normalizer(A.T @ Q)
/Users/lara/.pyenv/versions/3.11.8/lib/python3.11/site-packages/sklearn/utils/extmath.py:351: RuntimeWarning: invalid value encountered in 

    saved gallery for cluster 10 -> artifacts_posters/experiments/kmeans_k20_pca240/cluster_galleries/cluster_10.jpg
    saved gallery for cluster 16 -> artifacts_posters/experiments/kmeans_k20_pca240/cluster_galleries/cluster_16.jpg
    saved gallery for cluster 1 -> artifacts_posters/experiments/kmeans_k20_pca240/cluster_galleries/cluster_1.jpg
    saved gallery for cluster 14 -> artifacts_posters/experiments/kmeans_k20_pca240/cluster_galleries/cluster_14.jpg
    saved gallery for cluster 6 -> artifacts_posters/experiments/kmeans_k20_pca240/cluster_galleries/cluster_6.jpg
    saved gallery for cluster 5 -> artifacts_posters/experiments/kmeans_k20_pca240/cluster_galleries/cluster_5.jpg
    saved gallery for cluster 8 -> artifacts_posters/experiments/kmeans_k20_pca240/cluster_galleries/cluster_8.jpg
    saved gallery for cluster 15 -> artifacts_posters/experiments/kmeans_k20_pca240/cluster_galleries/cluster_15.jpg
    saved gallery for cluster 13 -> artifacts_posters/experiments/kmeans

/Users/lara/.pyenv/versions/3.11.8/lib/python3.11/site-packages/sklearn/utils/extmath.py:350: RuntimeWarning: divide by zero encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/lara/.pyenv/versions/3.11.8/lib/python3.11/site-packages/sklearn/utils/extmath.py:350: RuntimeWarning: overflow encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/lara/.pyenv/versions/3.11.8/lib/python3.11/site-packages/sklearn/utils/extmath.py:350: RuntimeWarning: invalid value encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/lara/.pyenv/versions/3.11.8/lib/python3.11/site-packages/sklearn/utils/extmath.py:351: RuntimeWarning: divide by zero encountered in matmul
  Q, _ = normalizer(A.T @ Q)
/Users/lara/.pyenv/versions/3.11.8/lib/python3.11/site-packages/sklearn/utils/extmath.py:351: RuntimeWarning: overflow encountered in matmul
  Q, _ = normalizer(A.T @ Q)
/Users/lara/.pyenv/versions/3.11.8/lib/python3.11/site-packages/sklearn/utils/extmath.py:351: RuntimeWarning: invalid value encountered in 

  wrote enriched metadata with clusters to: artifacts_posters/experiments/kmeans_k50_pca280/embeddings_with_clusters.json
    saved gallery for cluster 28 -> artifacts_posters/experiments/kmeans_k50_pca280/cluster_galleries/cluster_28.jpg
    saved gallery for cluster 48 -> artifacts_posters/experiments/kmeans_k50_pca280/cluster_galleries/cluster_48.jpg
    saved gallery for cluster 5 -> artifacts_posters/experiments/kmeans_k50_pca280/cluster_galleries/cluster_5.jpg
    saved gallery for cluster 16 -> artifacts_posters/experiments/kmeans_k50_pca280/cluster_galleries/cluster_16.jpg
    saved gallery for cluster 24 -> artifacts_posters/experiments/kmeans_k50_pca280/cluster_galleries/cluster_24.jpg
    saved gallery for cluster 6 -> artifacts_posters/experiments/kmeans_k50_pca280/cluster_galleries/cluster_6.jpg
    saved gallery for cluster 35 -> artifacts_posters/experiments/kmeans_k50_pca280/cluster_galleries/cluster_35.jpg
    saved gallery for cluster 31 -> artifacts_posters/experimen

/Users/lara/.pyenv/versions/3.11.8/lib/python3.11/site-packages/sklearn/utils/extmath.py:350: RuntimeWarning: divide by zero encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/lara/.pyenv/versions/3.11.8/lib/python3.11/site-packages/sklearn/utils/extmath.py:350: RuntimeWarning: overflow encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/lara/.pyenv/versions/3.11.8/lib/python3.11/site-packages/sklearn/utils/extmath.py:350: RuntimeWarning: invalid value encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/lara/.pyenv/versions/3.11.8/lib/python3.11/site-packages/sklearn/utils/extmath.py:351: RuntimeWarning: divide by zero encountered in matmul
  Q, _ = normalizer(A.T @ Q)
/Users/lara/.pyenv/versions/3.11.8/lib/python3.11/site-packages/sklearn/utils/extmath.py:351: RuntimeWarning: overflow encountered in matmul
  Q, _ = normalizer(A.T @ Q)
/Users/lara/.pyenv/versions/3.11.8/lib/python3.11/site-packages/sklearn/utils/extmath.py:351: RuntimeWarning: invalid value encountered in 

    saved gallery for cluster 1 -> artifacts_posters/experiments/kmeans_k20_pca120/cluster_galleries/cluster_1.jpg
    saved gallery for cluster 18 -> artifacts_posters/experiments/kmeans_k20_pca120/cluster_galleries/cluster_18.jpg
    saved gallery for cluster 6 -> artifacts_posters/experiments/kmeans_k20_pca120/cluster_galleries/cluster_6.jpg
    saved gallery for cluster 14 -> artifacts_posters/experiments/kmeans_k20_pca120/cluster_galleries/cluster_14.jpg
    saved gallery for cluster 17 -> artifacts_posters/experiments/kmeans_k20_pca120/cluster_galleries/cluster_17.jpg
    saved gallery for cluster 3 -> artifacts_posters/experiments/kmeans_k20_pca120/cluster_galleries/cluster_3.jpg
    saved gallery for cluster 19 -> artifacts_posters/experiments/kmeans_k20_pca120/cluster_galleries/cluster_19.jpg
    saved gallery for cluster 4 -> artifacts_posters/experiments/kmeans_k20_pca120/cluster_galleries/cluster_4.jpg
    saved gallery for cluster 8 -> artifacts_posters/experiments/kmeans_

/Users/lara/.pyenv/versions/3.11.8/lib/python3.11/site-packages/sklearn/utils/extmath.py:350: RuntimeWarning: divide by zero encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/lara/.pyenv/versions/3.11.8/lib/python3.11/site-packages/sklearn/utils/extmath.py:350: RuntimeWarning: overflow encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/lara/.pyenv/versions/3.11.8/lib/python3.11/site-packages/sklearn/utils/extmath.py:350: RuntimeWarning: invalid value encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/lara/.pyenv/versions/3.11.8/lib/python3.11/site-packages/sklearn/utils/extmath.py:351: RuntimeWarning: divide by zero encountered in matmul
  Q, _ = normalizer(A.T @ Q)
/Users/lara/.pyenv/versions/3.11.8/lib/python3.11/site-packages/sklearn/utils/extmath.py:351: RuntimeWarning: overflow encountered in matmul
  Q, _ = normalizer(A.T @ Q)
/Users/lara/.pyenv/versions/3.11.8/lib/python3.11/site-packages/sklearn/utils/extmath.py:351: RuntimeWarning: invalid value encountered in 

    saved gallery for cluster 21 -> artifacts_posters/experiments/kmeans_k40_pca160/cluster_galleries/cluster_21.jpg
    saved gallery for cluster 28 -> artifacts_posters/experiments/kmeans_k40_pca160/cluster_galleries/cluster_28.jpg
    saved gallery for cluster 12 -> artifacts_posters/experiments/kmeans_k40_pca160/cluster_galleries/cluster_12.jpg
    saved gallery for cluster 25 -> artifacts_posters/experiments/kmeans_k40_pca160/cluster_galleries/cluster_25.jpg
    saved gallery for cluster 30 -> artifacts_posters/experiments/kmeans_k40_pca160/cluster_galleries/cluster_30.jpg
    saved gallery for cluster 15 -> artifacts_posters/experiments/kmeans_k40_pca160/cluster_galleries/cluster_15.jpg
    saved gallery for cluster 24 -> artifacts_posters/experiments/kmeans_k40_pca160/cluster_galleries/cluster_24.jpg
    saved gallery for cluster 6 -> artifacts_posters/experiments/kmeans_k40_pca160/cluster_galleries/cluster_6.jpg
    saved gallery for cluster 35 -> artifacts_posters/experiments/

In [15]:
# saving the clusters I want to keep in a JSON file: artifacts_posters/experiments/selected_clusters.json
# the JSON looks like this:
{
  "experiment_name": "kmeans_k50_pca280_run1",
  "clusters": [
    { "cluster_id": 1,  "label": "Prestige / Arthouse Horror",                  "collage_file": "cluster_1.jpg" },
    { "cluster_id": 2,  "label": "HK–Taiwan Ghosts & Jiangshi",                "collage_file": "cluster_2.jpg" },
    { "cluster_id": 3,  "label": "Blue-Toned Alien / Underwater Isolation",    "collage_file": "cluster_3.jpg" },
    { "cluster_id": 5,  "label": "Shark & Aquatic Monsters",                   "collage_file": "cluster_5.jpg" },
    { "cluster_id": 7,  "label": "Haunted House / Atmospheric Supernatural",   "collage_file": "cluster_7.jpg" },
    { "cluster_id": 11, "label": "Japanese V-Cinema Horror",                   "collage_file": "cluster_11.jpg" },
    { "cluster_id": 12, "label": "Psychological / Trauma Arthouse",            "collage_file": "cluster_12.jpg" },
    { "cluster_id": 14, "label": "Zombies & ‘Dead’ Branding",                  "collage_file": "cluster_14.jpg" },
    { "cluster_id": 15, "label": "Torture Porn / Extreme Violence",            "collage_file": "cluster_15.jpg" },
    { "cluster_id": 17, "label": "Possessed Families & Haunted Homes",         "collage_file": "cluster_17.jpg" },
    { "cluster_id": 19, "label": "Illustrated Monsters / Psychedelic Cult",    "collage_file": "cluster_19.jpg" },
    { "cluster_id": 20, "label": "Elite J/K-Horror Curses",                    "collage_file": "cluster_20.jpg" },
    { "cluster_id": 22, "label": "Vintage Occult & Witchcraft",                "collage_file": "cluster_22.jpg" },
    { "cluster_id": 25, "label": "1950s Creature Features",                    "collage_file": "cluster_25.jpg" },
    { "cluster_id": 27, "label": "Anime Horror / Dark Fantasy",                "collage_file": "cluster_27.jpg" },
    { "cluster_id": 28, "label": "Modern Slasher & Gore Revival",              "collage_file": "cluster_28.jpg" },
    { "cluster_id": 31, "label": "Modern Curses & Ghost Stories",              "collage_file": "cluster_31.jpg" },
    { "cluster_id": 32, "label": "Ritual / Folk / Exploitation Horror",        "collage_file": "cluster_32.jpg" },
    { "cluster_id": 34, "label": "Vintage Japanese Kaidan",                    "collage_file": "cluster_34.jpg" },
    { "cluster_id": 35, "label": "VHS-Era Neon Supernatural Cult",             "collage_file": "cluster_35.jpg" },
    { "cluster_id": 37, "label": "Modern Kaiju & Giant Creatures",             "collage_file": "cluster_37.jpg" },
    { "cluster_id": 39, "label": "VHS-Era Cult Horror (Hybrid)",               "collage_file": "cluster_39.jpg" },
    { "cluster_id": 49, "label": "Religious Horror / Exorcism",                "collage_file": "cluster_49.jpg" }
  ]
}


{'experiment_name': 'kmeans_k50_pca280_run1',
 'clusters': [{'cluster_id': 1,
   'label': 'Prestige / Arthouse Horror',
   'collage_file': 'cluster_1.jpg'},
  {'cluster_id': 2,
   'label': 'HK–Taiwan Ghosts & Jiangshi',
   'collage_file': 'cluster_2.jpg'},
  {'cluster_id': 3,
   'label': 'Blue-Toned Alien / Underwater Isolation',
   'collage_file': 'cluster_3.jpg'},
  {'cluster_id': 5,
   'label': 'Shark & Aquatic Monsters',
   'collage_file': 'cluster_5.jpg'},
  {'cluster_id': 7,
   'label': 'Haunted House / Atmospheric Supernatural',
   'collage_file': 'cluster_7.jpg'},
  {'cluster_id': 11,
   'label': 'Japanese V-Cinema Horror',
   'collage_file': 'cluster_11.jpg'},
  {'cluster_id': 12,
   'label': 'Psychological / Trauma Arthouse',
   'collage_file': 'cluster_12.jpg'},
  {'cluster_id': 14,
   'label': 'Zombies & ‘Dead’ Branding',
   'collage_file': 'cluster_14.jpg'},
  {'cluster_id': 15,
   'label': 'Torture Porn / Extreme Violence',
   'collage_file': 'cluster_15.jpg'},
  {'cluste

In [ ]:
# now we want to define the centers for the selected clusters,
# to get one 512-dimensional vector per cluster, which becomes the "prototype" of that cluster

import json, numpy as np

# 1) Load cluster selection
with open("artifacts_posters/experiments/selected_clusters.json") as f:
    selected_ids = {c["cluster_id"] for c in json.load(f)["clusters"]}

# 2) Compute normalized mean vector per cluster
cluster_prototypes = {}

for cid in selected_ids:
    vecs = [
        info["embedding"] / np.linalg.norm(info["embedding"])
        for info in index_with_clusters
        if info.get("cluster") == cid
    ]
    if vecs:
        center = np.mean(vecs, axis=0)
        center = center / np.linalg.norm(center)
        cluster_prototypes[cid] = center

# 3) Save
np.save("cluster_prototypes.npy", cluster_prototypes)

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [20]:
with open("artifacts_posters/experiments/selected_clusters.json", "r", encoding="utf-8-sig") as f:
    txt = f.read()
    print(repr(txt[:80]))


''
